# Survival Modeling

Objective: modeling the survivability of the patients using the clinical dataset.

## Importing Modules

In [ ]:
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.impute import SimpleImputer

from leukemia.models import evaluation
from leukemia.data import loader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import  make_pipeline
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sklearn.impute import SimpleImputer

## Importing data

In [ ]:
X, y = loader.load_clinical_processed("../data/processed/clinical_train_processed_v1.csv")
num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG', 'BM_BLAST_MISSING', 'WBC_MISSING',
       'ANC_MISSING', 'MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING',
       'CYTO_NORMAL', 'CYTO_COMPLEX', 'CYTO_ABNORMAL', 'CYTO_MISSING']

## 1. Baseline: LightGBM

In [ ]:
preprocess_LGB = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER']),
])

mod_pipe_LGB = LGBMRegressor()

pipe_LGB = make_pipeline(preprocess_LGB, mod_pipe_LGB)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scorer = evaluation.ipcw_c_index
scores = []

for fold, (train_index, val_index) in enumerate(cv.split(X, y["OS_STATUS"])):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    pipe_LGB.fit(X_train, y_train['OS_YEARS'])

    y_train = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])
    y_val = loader.make_surv_target(y_val['OS_YEARS'], y_val['OS_STATUS'])

    scores.append({"fold": fold, "train_sore": scorer(y_train, y_train, -pipe_LGB.predict(X_train)),
    "val_sore": scorer(y_train, y_val, -pipe_LGB.predict(X_val))})

scores = pd.DataFrame(scores)
print(scores, scores["val_sore"].mean(), scores["val_sore"].std())

The LightGBM shows a c-index score of 0.617 ± 0.007.

fold  train_sore  val_sore
0     0    0.785393  0.614165
1     1    0.782646  0.624920
2     2    0.790919  0.622981
3     3    0.789521  0.608583
4     4    0.783311  0.615307 0.617191522081842 0.0067094894558244935

## 2. COX Regression

In [ ]:
preprocess_cox = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER'])])

pipe_cox = make_pipeline(preprocess_cox, CoxPHSurvivalAnalysis(alpha=1e-4))

scorer = evaluation.ipcw_c_index
scores = []

for fold, (train_index, val_index) in enumerate(cv.split(X, y["OS_STATUS"])):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    y_train = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])
    y_val = loader.make_surv_target(y_val['OS_YEARS'], y_val['OS_STATUS'])

    pipe_cox.fit(X_train, y_train)
    risk = pipe_cox.predict(X_val)

    scores.append({"fold": fold, "train_sore": scorer(y_train, y_train, pipe_cox.predict(X_train)),
    "val_sore": scorer(y_train, y_val, pipe_cox.predict(X_val))})

scores = pd.DataFrame(scores)
print(scores, scores["val_sore"].mean(), scores["val_sore"].std())


The Cox regression shows a mean c-index scores of 0.6861337507128616 ± 0.0064.
fold  train_sore  val_sore
0     0    0.696597  0.675957
1     1    0.690912  0.684018
2     2    0.692309  0.688814
3     3    0.694176  0.689705
4     4    0.692389  0.692175 0.6861337507128616 0.006412489226689529

Studying the model coefficients.

In [ ]:
coefs = pd.Series(
    pipe_cox[-1].coef_,
    index=pipe_cox[0].get_feature_names_out(),
).sort_values()
coefs

The coefficients are:
cat__CENTER_TUD          -0.620789
cat__CENTER_ROM          -0.533133
cat__CENTER_CCH          -0.420067
cat__CENTER_PV           -0.399919
num__HB                  -0.275372
num__PLT_LOG             -0.257155
num__CYTO_NORMAL         -0.146335
cat__CENTER_CGM          -0.141037
num__WBC_MISSING         -0.120924
cat__CENTER_UOB          -0.117008
cat__CENTER_VU           -0.114147
num__HB_MISSING          -0.060224
num__PLT_MISSING         -0.059266
cat__CENTER_MSK          -0.036125
num__WBC_LOG             -0.031462
cat__CENTER_RMCN         -0.027021
cat__CENTER_HMS          -0.019382
cat__CENTER_KI           -0.006208
cat__CENTER_REL           0.014416
num__MONOCYTES_MISSING    0.029271
cat__CENTER_UMG           0.031023
num__CYTO_COMPLEX         0.050990
cat__CENTER_GESMD         0.054614
num__ANC_MISSING          0.074596
num__ANC_LOG              0.077469
num__CYTO_MISSING         0.081304
num__MONOCYTES_LOG        0.090538
cat__CENTER_DUTH          0.092985
cat__CENTER_ICO           0.096350
num__CYTO_ABNORMAL        0.098325
num__BM_BLAST_MISSING     0.120258
cat__CENTER_IHBT          0.134854
cat__CENTER_FUCE          0.147524
cat__CENTER_FLO           0.186773
cat__CENTER_DUS           0.225680
cat__CENTER_UOXF          0.363862
num__BM_BLAST_LOG         0.397827
cat__CENTER_HIAE          0.444672
cat__CENTER_MUV           0.642084